# IAN 620 · Complete LangGraph Tutorial
## State, Nodes, Edges, Routing, Reducers, Streaming, Persistence, Human-in-the-Loop, Tools, and Agentic RAG

This notebook is intentionally **LangGraph-focused**. It explains not just the syntax, but why each graph primitive exists, what its important parameters mean, what functions return, and when you should choose one pattern over another.

### Learning goals

By the end, you should be able to:

1. Explain what LangGraph is and when to use it.
2. Design a state schema before writing nodes.
3. Build nodes that return partial state updates.
4. Connect nodes with normal and conditional edges.
5. Use reducers when multiple updates must be combined.
6. Build loops safely with explicit stopping conditions.
7. Use `Command` for combined update + routing.
8. Use `Send` for dynamic fan-out / map-reduce.
9. Stream graph updates, values, messages, and custom events.
10. Persist graph state with a checkpointer and `thread_id`.
11. Inspect, modify, and replay state.
12. Pause and resume with `interrupt()`.
13. Use runtime context separately from graph state.
14. Configure node retries.
15. Use `MessagesState`, `ToolNode`, and `tools_condition`.
16. Build a complete corrective RAG graph.
17. Understand the Functional API (`@entrypoint`, `@task`) as an alternative to `StateGraph`.

> The first half requires **no LLM and no API key**. Later sections use a downloadable local Hugging Face model.

## 0. What is LangGraph?

LangGraph is a low-level orchestration framework for **stateful, multi-step workflows and agents**.

The key idea is simple:

**State + Nodes + Edges = Graph**

- **State** is the shared working record.
- **Nodes** are functions that do work and return state updates.
- **Edges** decide which node runs next.

LangGraph becomes especially useful when an application needs:

- branching,
- loops,
- retries,
- persistence,
- streaming,
- human approval,
- long-running execution,
- a mix of deterministic code and LLM decisions.

### LangChain vs. LangGraph

| Need | Better starting point |
|---|---|
| Prompt/model integration | LangChain |
| Documents, embeddings, retrievers, tools | LangChain |
| Standard model ↔ tool agent loop | LangChain `create_agent()` |
| Explicit state machine | LangGraph |
| Branching and correction loops | LangGraph |
| Persistence / checkpoints | LangGraph |
| Human-in-the-loop | LangGraph |
| Complex deterministic + agentic workflow | LangGraph |

LangGraph can use LangChain components inside nodes, but it does not require LangChain.

Current LangChain agents are themselves built on LangGraph, but you do not need LangGraph for a basic LangChain agent.

### The design process

Before writing LangGraph code, ask:

1. **What process am I automating?**
2. **What are the distinct steps?** → nodes
3. **What information must survive across steps?** → state
4. **Where are the decisions?** → conditional edges or `Command`
5. **Where can the workflow repeat?** → cycles
6. **How does it stop?** → `END` / stopping conditions
7. **What can fail transiently?** → retry policies
8. **Where might a human need to intervene?** → `interrupt()`

A good graph starts as a process diagram, not as framework syntax.

## 1. Install packages

The first half of the notebook needs only LangGraph and LangChain Core. Later sections add a local model, embeddings, and Chroma for realistic examples.

In [ ]:
!pip -q install -U \
    langgraph \
    langchain-core \
    langchain-huggingface \
    langchain-chroma \
    langchain-text-splitters \
    chromadb \
    "sentence-transformers>=5.2.0" \
    "transformers>=4.52.4" \
    accelerate \
    pandas

## 2. Imports

`StateGraph`, `START`, and `END` are the central Graph API imports.

In [ ]:
import operator
import shutil
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langgraph.graph import (
    StateGraph,
    START,
    END,
    MessagesState,
)

from langgraph.types import (
    Command,
    Send,
    RetryPolicy,
    interrupt,
)

print('LangGraph imports ready.')

# Part I — StateGraph Fundamentals

## 3. `StateGraph`

`StateGraph` is a **builder** for a graph whose nodes read and write shared state.

Important idea:

**A `StateGraph` is not executable until you call `.compile()`.**

### Main constructor concepts

| Argument | Meaning |
|---|---|
| `state_schema` | Schema describing persistent graph state |
| `context_schema` | Optional schema for run-scoped context that should not be stored as graph state |

A node normally follows this contract:

`State -> Partial<State>`

The node reads the full state but usually returns only the keys it changes.

## 4. Define state with `TypedDict`

`TypedDict` is a common way to describe graph state.

Use state for values that must survive across nodes.

Do **not** automatically put every local variable into state.

In [ ]:
class SimpleState(TypedDict):
    text: str
    length: int
    label: str

### What belongs in state?

Store a value when a later node needs it or when it is useful for persistence/debugging.

Good state:

- original user request,
- retrieved evidence,
- classification result,
- retry/step count,
- draft answer,
- approval status.

Usually avoid storing:

- prompt strings that can be rebuilt,
- temporary formatting,
- duplicate values,
- values used only inside one node.

## 5. Nodes are ordinary functions

A node receives the current state and returns a dictionary containing updates.

### Important rule

Return **updates**, not a mutated global object.

This makes the graph easier to reason about, persist, replay, and test.

In [ ]:
def measure_text(state: SimpleState):
    return {
        'length': len(state['text'])
    }


def classify_length(state: SimpleState):
    label = (
        'long'
        if state['length'] >= 20
        else 'short'
    )

    return {
        'label': label
    }

## 6. `add_node()`

`add_node()` registers a function or Runnable as a graph node.

### Common parameters

| Parameter | Meaning |
|---|---|
| `node` | Node function/Runnable, or node name when using the two-argument form |
| `action` | Function/Runnable when the first argument is a custom name |
| `input_schema` | Optional node-specific input schema |
| `retry_policy` | Retry configuration for this node |
| `cache_policy` | Optional cache policy |
| `metadata` | Metadata associated with the node |
| `timeout` | Node timeout policy; async execution supports cancellable timeouts |
| `defer` | Delay this node until the run is about to finish |

Two common styles:

```python
builder.add_node(my_function)
builder.add_node('custom_name', my_function)
```

In [ ]:
builder = StateGraph(SimpleState)

builder.add_node(
    'measure',
    measure_text,
)

builder.add_node(
    'classify',
    classify_length,
)

## 7. `START` and `END`

`START` and `END` are virtual graph nodes.

- `START` represents graph entry.
- `END` represents graph termination.

Use them to make the graph's boundaries explicit.

## 8. `add_edge()`

`add_edge(start_key, end_key)` adds an unconditional transition.

Examples:

```python
builder.add_edge(START, 'measure')
builder.add_edge('measure', 'classify')
builder.add_edge('classify', END)
```

If `start_key` is a **list of node names**, LangGraph waits for all of those nodes to complete before running the destination node. This is useful for joins after parallel work.

In [ ]:
builder.add_edge(START, 'measure')
builder.add_edge('measure', 'classify')
builder.add_edge('classify', END)

## 9. `.compile()`

`compile()` converts the builder into an executable `CompiledStateGraph`.

### Important `compile()` options

| Parameter | Purpose |
|---|---|
| `checkpointer` | Persist graph state across steps/runs |
| `interrupt_before` | Pause before selected nodes |
| `interrupt_after` | Pause after selected nodes |
| `debug` | Enable graph debug mode |
| `name` | Optional compiled graph name |

The compiled graph implements the Runnable interface, so it supports `.invoke()`, `.stream()`, `.batch()`, async variants, and more.

In [ ]:
simple_graph = builder.compile(
    name='simple_text_graph'
)

result = simple_graph.invoke({
    'text': 'LangGraph keeps shared state.',
    'length': 0,
    'label': '',
})

print(result)

## 10. Inspect the graph

`get_graph()` exposes a structural representation of the compiled graph.

`draw_mermaid()` is convenient in notebooks because it produces Mermaid syntax without requiring image rendering.

In [ ]:
print(simple_graph.get_graph().draw_mermaid())

### Builder vs. compiled graph

| Object | Purpose |
|---|---|
| `StateGraph(...)` | Define nodes/edges/schema |
| `CompiledStateGraph` | Execute, stream, persist, inspect state |

Do not try to call `.invoke()` on the builder.

# Part II — State Updates and Reducers

## 11. Default state behavior: last update wins

For a normal state key, a node update replaces the previous value.

Example:

`{'status': 'new'} → node returns {'status': 'done'} → state becomes 'done'`

This is correct for scalar fields such as status, route, score, or answer.

## 12. Why reducers exist

Reducers are needed when multiple updates to the same state key should be **combined** rather than replaced.

Common cases:

- accumulating logs,
- collecting results from parallel workers,
- appending messages,
- aggregating map-reduce outputs.

A reducer has the conceptual signature:

`(current_value, new_value) -> combined_value`

## 13. `Annotated[..., operator.add]`

For lists, `operator.add` is a simple append/concatenate reducer.

In [ ]:
class LogState(TypedDict):
    value: int
    log: Annotated[list[str], operator.add]


def first_step(state: LogState):
    return {
        'value': state['value'] + 1,
        'log': ['first_step'],
    }


def second_step(state: LogState):
    return {
        'value': state['value'] * 2,
        'log': ['second_step'],
    }


log_builder = StateGraph(LogState)
log_builder.add_node('first', first_step)
log_builder.add_node('second', second_step)
log_builder.add_edge(START, 'first')
log_builder.add_edge('first', 'second')
log_builder.add_edge('second', END)

log_graph = log_builder.compile()

log_graph.invoke({
    'value': 2,
    'log': [],
})

## 14. Custom reducers

Use a custom reducer when merging needs domain logic.

Requirements for good reducers:

- deterministic,
- associative across repeated folds,
- predictable when replayed from checkpoints.

Example: keep unique strings while preserving order.

In [ ]:
def unique_merge(
    current: list[str],
    new: list[str],
) -> list[str]:
    result = list(current)

    for item in new:
        if item not in result:
            result.append(item)

    return result


class UniqueState(TypedDict):
    items: Annotated[list[str], unique_merge]

## 15. Parallel branches

Multiple edges from the same source can make nodes runnable in the same graph step.

If parallel nodes update the **same normal state key**, LangGraph may raise an invalid concurrent update error because it does not know how to combine the writes.

Use a reducer for state keys intentionally written by parallel branches.

In [ ]:
class ParallelState(TypedDict):
    number: int
    results: Annotated[list[str], operator.add]


def square_branch(state: ParallelState):
    return {
        'results': [
            f"square={state['number'] ** 2}"
        ]
    }


def cube_branch(state: ParallelState):
    return {
        'results': [
            f"cube={state['number'] ** 3}"
        ]
    }


def summarize_parallel(state: ParallelState):
    return {
        'results': [
            'joined both branches'
        ]
    }

In [ ]:
parallel_builder = StateGraph(ParallelState)
parallel_builder.add_node('square', square_branch)
parallel_builder.add_node('cube', cube_branch)
parallel_builder.add_node('join', summarize_parallel)

parallel_builder.add_edge(START, 'square')
parallel_builder.add_edge(START, 'cube')

# A list here means: wait until BOTH nodes finish.
parallel_builder.add_edge(
    ['square', 'cube'],
    'join',
)

parallel_builder.add_edge('join', END)

parallel_graph = parallel_builder.compile()

parallel_graph.invoke({
    'number': 3,
    'results': [],
})

# Part III — Conditional Routing

## 16. `add_conditional_edges()`

`add_conditional_edges()` runs a routing function after a source node and chooses the next destination.

### Parameters

| Parameter | Meaning |
|---|---|
| `source` | Node whose completion triggers routing |
| `path` | Function/Runnable that returns a route key or destination |
| `path_map` | Optional map from route keys to node names |

A router should usually be simple and easy to test.

In [ ]:
class RouteState(TypedDict):
    number: int
    route: str
    result: str


def inspect_number(state: RouteState):
    return {
        'route': (
            'even'
            if state['number'] % 2 == 0
            else 'odd'
        )
    }


def choose_route(
    state: RouteState,
) -> Literal['even', 'odd']:
    return state['route']


def even_node(state: RouteState):
    return {'result': 'Number is even'}


def odd_node(state: RouteState):
    return {'result': 'Number is odd'}

In [ ]:
route_builder = StateGraph(RouteState)
route_builder.add_node('inspect', inspect_number)
route_builder.add_node('even_node', even_node)
route_builder.add_node('odd_node', odd_node)

route_builder.add_edge(START, 'inspect')

route_builder.add_conditional_edges(
    'inspect',
    choose_route,
    {
        'even': 'even_node',
        'odd': 'odd_node',
    },
)

route_builder.add_edge('even_node', END)
route_builder.add_edge('odd_node', END)

route_graph = route_builder.compile()

print(route_graph.invoke({
    'number': 8,
    'route': '',
    'result': '',
}))

### `path_map` vs. direct node names

Without `path_map`, your routing function can return actual node names.

With `path_map`, the router can return semantic labels such as `approve` or `reject`, while the graph maps those labels to implementation node names.

This can make routing logic easier to read and test.

## 17. Conditional entry points

LangGraph also supports conditional routing directly from the beginning of a graph.

You can use:

- `add_conditional_edges(START, router, ...)`, or
- `set_conditional_entry_point(...)`.

The `START` form keeps the graph structure visually explicit and is easy to compare with ordinary edges.

# Part IV — Cycles, Limits, and Safe Loops

## 18. Cycles are allowed

Unlike a simple pipeline, a graph can return to an earlier node.

Cycles are useful for:

- retrying retrieval,
- revising a draft,
- tool-use loops,
- planning/execution loops,
- iterative validation.

But every cycle needs an explicit stopping policy.

In [ ]:
class LoopState(TypedDict):
    count: int
    history: Annotated[list[int], operator.add]


def increment(state: LoopState):
    new_count = state['count'] + 1
    return {
        'count': new_count,
        'history': [new_count],
    }


def continue_or_stop(
    state: LoopState,
) -> Literal['again', 'stop']:
    return (
        'again'
        if state['count'] < 3
        else 'stop'
    )

In [ ]:
loop_builder = StateGraph(LoopState)
loop_builder.add_node('increment', increment)
loop_builder.add_edge(START, 'increment')

loop_builder.add_conditional_edges(
    'increment',
    continue_or_stop,
    {
        'again': 'increment',
        'stop': END,
    },
)

loop_graph = loop_builder.compile()

loop_graph.invoke({
    'count': 0,
    'history': [],
})

## 19. `recursion_limit`

`recursion_limit` is a runtime safety net that limits graph steps/cycles.

Example:

```python
graph.invoke(
    input_state,
    {"recursion_limit": 10},
)
```

LangGraph's runtime has a default recursion limit, but application-level stop conditions are still important.

### Best practice

Use both:

1. an **application-specific limit** in state/logic, and
2. `recursion_limit` as a final runtime guard.

In [ ]:
safe_loop_result = loop_graph.invoke(
    {
        'count': 0,
        'history': [],
    },
    {
        'recursion_limit': 10
    },
)

print(safe_loop_result)

# Part V — `Command`: Update State and Route Together

## 20. What is `Command`?

`Command` can combine graph actions in one return value.

Important fields:

| Field | Purpose |
|---|---|
| `update` | State update to apply |
| `goto` | Destination node(s) |
| `resume` | Resume value for an `interrupt()` |
| `graph` | Target graph; `Command.PARENT` can target a parent graph |

`Command` is especially useful when a node naturally decides both:

**what state changed** and **where execution should go next**.

In [ ]:
class CommandState(TypedDict):
    score: int
    status: str


def decision_node(
    state: CommandState,
) -> Command[Literal['pass_node', 'fail_node']]:
    if state['score'] >= 70:
        return Command(
            update={'status': 'passed'},
            goto='pass_node',
        )

    return Command(
        update={'status': 'failed'},
        goto='fail_node',
    )


def pass_node(state: CommandState):
    return {'status': state['status'] + ' → complete'}


def fail_node(state: CommandState):
    return {'status': state['status'] + ' → review'}

In [ ]:
command_builder = StateGraph(CommandState)
command_builder.add_node(
    'decision',
    decision_node,
    destinations=('pass_node', 'fail_node'),
)
command_builder.add_node('pass_node', pass_node)
command_builder.add_node('fail_node', fail_node)

command_builder.add_edge(START, 'decision')
command_builder.add_edge('pass_node', END)
command_builder.add_edge('fail_node', END)

command_graph = command_builder.compile()

command_graph.invoke({
    'score': 82,
    'status': '',
})

### `Command` vs. conditional edges

| Pattern | Good when |
|---|---|
| `add_conditional_edges()` | You want routing logic separated from node logic |
| `Command(goto=...)` | The node naturally decides its update and destination together |

Both are valid. Choose the form that makes the workflow easiest to inspect and test.

# Part VI — Dynamic Fan-Out with `Send`

## 21. What is `Send`?

`Send` dynamically schedules a node with custom input.

It is useful when the number of parallel tasks is not known until runtime.

Classic example: **map-reduce**.

1. Read a list.
2. Send one task per item.
3. Run the worker several times in parallel.
4. Use a reducer to collect results.

In [ ]:
class MapState(TypedDict):
    topics: list[str]
    summaries: Annotated[list[str], operator.add]


class WorkerState(TypedDict):
    topic: str


def fan_out(state: MapState):
    return [
        Send(
            'worker',
            {'topic': topic},
        )
        for topic in state['topics']
    ]


def worker(state: WorkerState):
    return {
        'summaries': [
            f"Summary placeholder for {state['topic']}"
        ]
    }

In [ ]:
map_builder = StateGraph(MapState)
map_builder.add_node('worker', worker)

map_builder.add_conditional_edges(
    START,
    fan_out,
)

map_builder.add_edge('worker', END)

map_graph = map_builder.compile()

map_graph.invoke({
    'topics': ['RAG', 'embeddings', 'agents'],
    'summaries': [],
})

### Why the reducer matters here

Each worker writes to `summaries` independently.

`Annotated[list[str], operator.add]` tells LangGraph how to merge those parallel writes.

Without a reducer, parallel workers writing to the same key would conflict.

# Part VII — Streaming

## 22. `.stream()`

Compiled graphs can stream execution rather than returning only the final state.

Important stream modes:

| `stream_mode` | What you receive |
|---|---|
| `updates` | Node/task state updates after each step |
| `values` | Full state after each step |
| `messages` | LLM message tokens plus metadata |
| `custom` | Custom values emitted by your code |
| `checkpoints` | Checkpoint events |
| `tasks` | Task start/finish events |
| `debug` | Detailed checkpoint + task debugging information |

## 23. Stream node updates

`updates` is often the easiest mode for understanding graph execution because it shows what each node returned.

In [ ]:
for chunk in simple_graph.stream(
    {
        'text': 'Streaming shows graph progress.',
        'length': 0,
        'label': '',
    },
    stream_mode='updates',
):
    print(chunk)

## 24. Stream full state values

`values` emits the complete state after graph steps.

In [ ]:
for chunk in simple_graph.stream(
    {
        'text': 'Full state example.',
        'length': 0,
        'label': '',
    },
    stream_mode='values',
):
    print(chunk)

## 25. Custom streaming with `get_stream_writer()`

Custom streaming is useful for progress messages that do not belong in persistent graph state.

Inside a node, call `get_stream_writer()` and emit any serializable value.

In [ ]:
from langgraph.config import get_stream_writer


class StreamState(TypedDict):
    total: int


def progress_node(state: StreamState):
    writer = get_stream_writer()

    writer({'stage': 1, 'message': 'starting'})
    writer({'stage': 2, 'message': 'computing'})

    return {
        'total': state['total'] + 10
    }


stream_builder = StateGraph(StreamState)
stream_builder.add_node('progress', progress_node)
stream_builder.add_edge(START, 'progress')
stream_builder.add_edge('progress', END)

stream_graph = stream_builder.compile()

In [ ]:
for event in stream_graph.stream(
    {'total': 0},
    stream_mode='custom',
):
    print(event)

### Multiple stream modes

You can pass more than one mode.

When multiple modes are requested, streamed items include the mode together with its data.

In [ ]:
for mode, event in stream_graph.stream(
    {'total': 0},
    stream_mode=['updates', 'custom'],
):
    print(mode, '→', event)

# Part VIII — Persistence and Checkpoints

## 26. Why checkpointing matters

A checkpointer stores versions of graph state.

This enables:

- conversation/thread memory,
- pause/resume,
- human-in-the-loop,
- inspecting state after execution,
- state history,
- replay/time-travel workflows.

With a checkpointer, you normally invoke the graph with:

```python
config = {"configurable": {"thread_id": "thread-1"}}
```

`thread_id` is the primary key for a line of checkpoint history.

## 27. `InMemorySaver`

`InMemorySaver` is convenient for notebooks and tests.

It is **not** a durable production database because the process memory disappears when the runtime stops.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver


class MemoryState(TypedDict):
    input_text: str
    history: Annotated[list[str], operator.add]


def remember(state: MemoryState):
    return {
        'history': [state['input_text']]
    }


memory_builder = StateGraph(MemoryState)
memory_builder.add_node('remember', remember)
memory_builder.add_edge(START, 'remember')
memory_builder.add_edge('remember', END)

memory = InMemorySaver()

memory_graph = memory_builder.compile(
    checkpointer=memory
)

## 28. `thread_id`

Use a different `thread_id` for independent conversations or runs.

Reuse a `thread_id` when later invocations should belong to the same persisted thread.

In [ ]:
thread_config = {"configurable": {"thread_id": "demo-thread"}}

print(memory_graph.invoke(
    {
        'input_text': 'first message',
        'history': [],
    },
    thread_config,
))

print(memory_graph.invoke(
    {
        'input_text': 'second message',
        'history': [],
    },
    thread_config,
))

## 29. `get_state()`

`get_state(config)` returns a `StateSnapshot`.

Useful snapshot fields include:

| Field | Meaning |
|---|---|
| `.values` | Current graph state |
| `.next` | Node(s) scheduled next |
| `.config` | Checkpoint configuration |
| `.metadata` | Checkpoint metadata |
| `.created_at` | Timestamp |
| `.tasks` | Tasks in this step |
| `.interrupts` | Outstanding interrupts |

In [ ]:
snapshot = memory_graph.get_state(
    thread_config
)

print('Values:', snapshot.values)
print('Next:', snapshot.next)
print('Created:', snapshot.created_at)

## 30. `get_state_history()`

`get_state_history(config)` iterates checkpoint snapshots for a thread.

This is useful for debugging and time-travel style inspection.

In [ ]:
history = list(
    memory_graph.get_state_history(
        thread_config
    )
)

print('Checkpoint count:', len(history))

for snap in history[:3]:
    print(snap.created_at, snap.values)

## 31. `update_state()`

`update_state(config, values, ...)` writes a state update as if a graph node produced it.

Useful cases:

- human correction,
- debugging,
- repairing a checkpoint,
- controlled time-travel workflows.

Important parameters:

| Parameter | Meaning |
|---|---|
| `config` | Thread/checkpoint to update |
| `values` | State values to write |
| `as_node` | Optionally treat update as if it came from a particular node |

In [ ]:
new_config = memory_graph.update_state(
    thread_config,
    {
        'history': ['manual correction']
    },
)

print(
    memory_graph.get_state(
        new_config
    ).values
)

# Part IX — Human-in-the-Loop with `interrupt()`

## 32. `interrupt()`

`interrupt(value)` pauses graph execution from inside a node and surfaces `value` to the caller.

To resume, invoke the graph with:

`Command(resume=...)`

### Requirement

A checkpointer must be enabled because the graph needs saved state while it is paused.

### Important interrupt behavior

When a graph resumes, the interrupted node starts again from the beginning.

Therefore:

- avoid non-idempotent side effects before `interrupt()`, or
- move them to a separate node, or
- make them safely repeatable.

This is one of the most important design rules for human-in-the-loop graphs.

In [ ]:
class ApprovalState(TypedDict):
    draft: str
    approved: bool
    status: str


def approval_node(state: ApprovalState):
    decision = interrupt({
        'question': 'Approve this draft?',
        'draft': state['draft'],
    })

    return {
        'approved': bool(decision)
    }


def finalize(state: ApprovalState):
    return {
        'status': (
            'approved'
            if state['approved']
            else 'rejected'
        )
    }

In [ ]:
approval_builder = StateGraph(ApprovalState)
approval_builder.add_node('approval', approval_node)
approval_builder.add_node('finalize', finalize)
approval_builder.add_edge(START, 'approval')
approval_builder.add_edge('approval', 'finalize')
approval_builder.add_edge('finalize', END)

approval_memory = InMemorySaver()

approval_graph = approval_builder.compile(
    checkpointer=approval_memory
)

approval_config = {"configurable": {"thread_id": "approval-1"}}

In [ ]:
paused = approval_graph.invoke(
    {
        'draft': 'Proposed response text',
        'approved': False,
        'status': 'waiting',
    },
    approval_config,
)

print(paused)

In [ ]:
resumed = approval_graph.invoke(
    Command(resume=True),
    approval_config,
)

print(resumed)

### Typed resume values

Current `interrupt()` can also accept a `response_schema` describing the expected resume value.

This is useful when a UI should render or validate a structured human response.

# Part X — Runtime Context

## 33. State vs. runtime context

Graph **state** is part of the workflow's evolving data.

Runtime **context** is static information/dependencies supplied for one graph run.

Good context examples:

- user ID,
- tenant ID,
- database connection,
- environment information,
- permissions.

Do not put every dependency into persistent state.

In [ ]:
from langgraph.runtime import Runtime


@dataclass
class UserContext:
    user_id: str
    role: str


class ContextState(TypedDict):
    greeting: str


def contextual_greeting(
    state: ContextState,
    runtime: Runtime[UserContext],
):
    return {
        'greeting': (
            f"Hello {runtime.context.user_id}. "
            f"Role: {runtime.context.role}."
        )
    }

In [ ]:
context_builder = StateGraph(
    ContextState,
    context_schema=UserContext,
)

context_builder.add_node(
    'greet',
    contextual_greeting,
)
context_builder.add_edge(START, 'greet')
context_builder.add_edge('greet', END)

context_graph = context_builder.compile()

context_graph.invoke(
    {'greeting': ''},
    context=UserContext(
        user_id='user-42',
        role='analyst',
    ),
)

### `Runtime`

A node can receive `Runtime[ContextType]`.

Runtime can expose:

- `.context` — static run-scoped context,
- `.store` — persistent store when configured,
- `.stream_writer` — custom stream writer,
- execution information.

If you need `RunnableConfig`, inject a separate `config: RunnableConfig` parameter rather than expecting it inside `Runtime`.

# Part XI — Retry Policies and Fault Tolerance

## 34. `RetryPolicy`

Retries belong on operations that can fail **transiently**, such as network calls.

Do not retry deterministic programming errors.

### Important parameters

| Parameter | Meaning |
|---|---|
| `initial_interval` | Delay before first retry |
| `backoff_factor` | Multiplier applied to retry delay |
| `max_interval` | Maximum delay between retries |
| `max_attempts` | Maximum attempts including the first |
| `jitter` | Add randomness to reduce synchronized retries |
| `retry_on` | Exception class/list/callable deciding what should retry |

In [ ]:
attempt_counter = {'count': 0}


class RetryState(TypedDict):
    result: str


def unstable_node(state: RetryState):
    attempt_counter['count'] += 1

    if attempt_counter['count'] < 3:
        raise ConnectionError(
            'temporary failure'
        )

    return {
        'result': 'success after retry'
    }


retry_builder = StateGraph(RetryState)

retry_builder.add_node(
    'unstable',
    unstable_node,
    retry_policy=RetryPolicy(
        initial_interval=0.01,
        backoff_factor=1.0,
        max_interval=0.01,
        max_attempts=3,
        jitter=False,
        retry_on=ConnectionError,
    ),
)

retry_builder.add_edge(START, 'unstable')
retry_builder.add_edge('unstable', END)

retry_graph = retry_builder.compile()

retry_graph.invoke({'result': ''})

## 35. `set_node_defaults()`

When many nodes need the same retry/timeout policy, a graph can define defaults.

Per-node settings still override graph defaults.

Conceptual example:

```python
builder.set_node_defaults(
    retry_policy=RetryPolicy(max_attempts=3),
)
```

This is useful when most external-I/O nodes share the same resilience policy.

# Part XII — Message State

## 36. `MessagesState`

`MessagesState` is a predefined state schema with one important field:

`messages`

The `messages` field uses LangGraph's `add_messages` reducer.

This reducer appends messages while also supporting message replacement by ID.

In [ ]:
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
)


def echo_message(state: MessagesState):
    last = state['messages'][-1]

    return {
        'messages': [
            AIMessage(
                content=f"Echo: {last.content}"
            )
        ]
    }


message_builder = StateGraph(MessagesState)
message_builder.add_node('echo', echo_message)
message_builder.add_edge(START, 'echo')
message_builder.add_edge('echo', END)

message_graph = message_builder.compile()

message_graph.invoke({
    'messages': [
        HumanMessage(content='Hello LangGraph')
    ]
})

## 37. `add_messages` reducer

Use `add_messages` when you define a custom state that includes messages plus other fields.

Unlike ordinary list concatenation, it understands message IDs and can update an existing message with the same ID.

In [ ]:
from langgraph.graph.message import add_messages


class CustomMessageState(TypedDict):
    messages: Annotated[list, add_messages]
    topic: str

# Part XIII — Tool Execution with `ToolNode`

## 38. `ToolNode`

`ToolNode` is a prebuilt LangGraph node that executes tool calls.

It handles:

- one or multiple tool calls,
- parallel execution when appropriate,
- conversion of outputs to `ToolMessage`,
- configurable tool-error handling,
- state/store/runtime injection patterns.

Use `ToolNode` when you are building a **custom graph** and want LangGraph to handle tool execution.

### Important `ToolNode` parameters

| Parameter | Meaning |
|---|---|
| `tools` | Sequence of tools or callables |
| `name` | Node name; default is `tools` |
| `tags` | Optional metadata tags |
| `handle_tool_errors` | Configure which tool errors are converted to tool messages |
| `messages_key` | State key containing messages |

For a standard ReAct-style agent, LangChain `create_agent()` is usually simpler. `ToolNode` is valuable when you need custom graph control.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


tool_node = ToolNode([multiply])

### Test `ToolNode` directly

We can create an `AIMessage` containing a tool call manually. This lets us test LangGraph tool execution without depending on a model.

In [ ]:
tool_call_message = AIMessage(
    content='',
    tool_calls=[
        {
            'name': 'multiply',
            'args': {'a': 6, 'b': 7},
            'id': 'call-1',
            'type': 'tool_call',
        }
    ],
)

tool_node.invoke({
    'messages': [tool_call_message]
})

## 39. `tools_condition`

`tools_condition` implements a standard routing rule:

- if the last `AIMessage` contains tool calls → route to `tools`,
- otherwise → route to `END`.

It is a convenience function for custom tool-calling graphs.

In [ ]:
from langgraph.prebuilt import tools_condition


def fake_model_node(state: MessagesState):
    last = state['messages'][-1]

    if isinstance(last, HumanMessage):
        return {
            'messages': [
                AIMessage(
                    content='',
                    tool_calls=[
                        {
                            'name': 'multiply',
                            'args': {'a': 4, 'b': 9},
                            'id': 'call-2',
                            'type': 'tool_call',
                        }
                    ],
                )
            ]
        }

    return {
        'messages': [
            AIMessage(
                content='Tool result received. Done.'
            )
        ]
    }

In [ ]:
tool_graph_builder = StateGraph(MessagesState)
tool_graph_builder.add_node('model', fake_model_node)
tool_graph_builder.add_node('tools', ToolNode([multiply]))

tool_graph_builder.add_edge(START, 'model')

tool_graph_builder.add_conditional_edges(
    'model',
    tools_condition,
    {
        'tools': 'tools',
        '__end__': END,
    },
)

tool_graph_builder.add_edge('tools', 'model')

tool_graph = tool_graph_builder.compile()

tool_graph.invoke({
    'messages': [
        HumanMessage(content='Multiply 4 by 9')
    ]
})

# Part XIV — Local LLM Inside LangGraph

## 40. Download a local model — no API key

LangGraph is model-agnostic. A model call is simply work performed inside a node.

We use a small Qwen model through LangChain's Hugging Face integration because it gives us a chat-model interface and requires no paid API key.

In [ ]:
import torch

from langchain_huggingface import (
    HuggingFacePipeline,
    ChatHuggingFace,
)

LOCAL_MODEL = (
    'Qwen/Qwen3-1.7B'
    if torch.cuda.is_available()
    else 'Qwen/Qwen3-0.6B'
)

hf_pipeline = HuggingFacePipeline.from_model_id(
    model_id=LOCAL_MODEL,
    task='text-generation',
    model_kwargs={
        'torch_dtype': 'auto',
    },
    pipeline_kwargs={
        'max_new_tokens': 180,
        'do_sample': False,
        'return_full_text': False,
    },
)

local_chat = ChatHuggingFace(
    llm=hf_pipeline
)

print('Loaded:', LOCAL_MODEL)

## 41. Put the model call inside a node

A model node is still just a node.

Keep prompts close to the node that uses them rather than storing formatted prompt text in state.

In [ ]:
class LLMState(TypedDict):
    question: str
    answer: str


def answer_node(state: LLMState):
    response = local_chat.invoke(
        'Answer in two short sentences: '
        + state['question']
    )

    return {
        'answer': response.content
    }


llm_builder = StateGraph(LLMState)
llm_builder.add_node('answer', answer_node)
llm_builder.add_edge(START, 'answer')
llm_builder.add_edge('answer', END)

llm_graph = llm_builder.compile()

llm_graph.invoke({
    'question': 'What is LangGraph?',
    'answer': '',
})

## 42. Streaming model messages

When a chat model runs inside a node, `stream_mode='messages'` can expose model message chunks together with metadata.

Exact streaming behavior depends on the underlying model integration.

In [ ]:
for message_chunk, metadata in llm_graph.stream(
    {
        'question': 'Explain graph state briefly.',
        'answer': '',
    },
    stream_mode='messages',
):
    if getattr(message_chunk, 'content', None):
        print(message_chunk.content, end='', flush=True)

# Part XV — Subgraphs

## 43. Why subgraphs?

A subgraph packages a multi-step workflow so it can be used as one node inside a larger graph.

Useful when:

- a workflow has a clear internal boundary,
- a sequence is reusable,
- you want hierarchical design,
- a team owns one part of a larger workflow.

In [ ]:
class SubState(TypedDict):
    value: int


def add_two(state: SubState):
    return {'value': state['value'] + 2}


def times_three(state: SubState):
    return {'value': state['value'] * 3}


sub_builder = StateGraph(SubState)
sub_builder.add_node('add_two', add_two)
sub_builder.add_node('times_three', times_three)
sub_builder.add_edge(START, 'add_two')
sub_builder.add_edge('add_two', 'times_three')
sub_builder.add_edge('times_three', END)

subgraph = sub_builder.compile()

In [ ]:
parent_builder = StateGraph(SubState)
parent_builder.add_node('math_subgraph', subgraph)
parent_builder.add_edge(START, 'math_subgraph')
parent_builder.add_edge('math_subgraph', END)

parent_graph = parent_builder.compile()

parent_graph.invoke({'value': 4})

### State compatibility

The simplest subgraph setup uses compatible state keys between parent and child.

If parent and child schemas differ, use a wrapper node that transforms parent state into the subgraph's input and maps its output back.

# Part XVI — Functional API

## 44. Graph API vs. Functional API

LangGraph offers two styles:

| API | Best when |
|---|---|
| Graph API (`StateGraph`) | You want explicit nodes, edges, state schemas, visual topology |
| Functional API (`@entrypoint`, `@task`) | You prefer ordinary Python control flow while still getting LangGraph execution features |

This notebook emphasizes `StateGraph`, but the Functional API is useful to recognize.

## 45. `@task` and `@entrypoint`

- `@task` marks a durable unit of work.
- `@entrypoint` defines a workflow entry function.

`@task` supports retry, cache, and timeout policies.

`@entrypoint` can also be compiled with checkpoint/store/context options.

In [ ]:
from langgraph.func import entrypoint, task


@task
def square_number(number: int) -> int:
    return number * number


@entrypoint()
def square_workflow(
    numbers: list[int],
) -> list[int]:
    futures = [
        square_number(n)
        for n in numbers
    ]

    return [
        future.result()
        for future in futures
    ]


square_workflow.invoke([2, 3, 4])

### Which API should students learn first?

For learning orchestration concepts, `StateGraph` is usually easier because the graph structure is visible.

The Functional API is valuable once you already understand:

- state/persistence,
- durable tasks,
- retries,
- interrupts,
- streaming.

# Part XVII — Complete Corrective RAG Graph

## 46. Build a realistic LangGraph workflow

Now we combine LangGraph with retrieval and a local model.

The goal is not to teach every RAG detail. The goal is to show how LangGraph orchestrates:

**retrieve → grade → generate OR rewrite → retrieve again**

The graph will use:

- CSV knowledge base,
- local Hugging Face embeddings,
- Chroma,
- local Qwen chat model,
- explicit LangGraph state and routing.

## 47. Load the knowledge-base dataset

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/ali2449/IAN-620/main/IAN_620_Module_9_Agentic_RAG_Knowledge_Base.csv"

kb = pd.read_csv(DATA_URL)

kb[['source_id', 'category', 'title']].head()

## 48. Convert rows to `Document` objects and build Chroma

In [ ]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

rag_documents = []

for _, row in kb.iterrows():
    rag_documents.append(
        Document(
            page_content=f"{row['title']}\n{row['text']}",
            metadata={
                'source_id': row['source_id'],
                'category': row['category'],
                'title': row['title'],
            },
        )
    )

rag_embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    encode_kwargs={
        'normalize_embeddings': True
    },
)

RAG_DIR = "./langgraph_rag_chroma"
shutil.rmtree(RAG_DIR, ignore_errors=True)

rag_store = Chroma.from_documents(
    documents=rag_documents,
    embedding=rag_embeddings,
    persist_directory=RAG_DIR,
    collection_name='langgraph_rag_demo',
)

rag_retriever = rag_store.as_retriever(
    search_kwargs={'k': 3}
)

## 49. Design the RAG graph state

Store only values needed across nodes.

`query` is separate from `question` because query rewriting may change the retrieval wording while preserving the user's original intent.

In [ ]:
class RAGState(TypedDict):
    question: str
    query: str
    documents: list[dict]
    score: float
    steps: int
    answer: str

## 50. Retrieval node

In [ ]:
def retrieve_node(state: RAGState):
    docs = rag_retriever.invoke(
        state['query']
    )

    serialized = [
        {
            'text': d.page_content,
            'source_id': d.metadata.get(
                'source_id',
                'unknown',
            ),
        }
        for d in docs
    ]

    return {
        'documents': serialized,
        'steps': state['steps'] + 1,
    }

## 51. Evidence-grading node

This deterministic grader is intentionally simple so the graph behavior is visible.

In a real system, the grader could be:

- a reranker,
- an LLM with structured output,
- a classifier,
- domain-specific rules.

In [ ]:
def grade_node(state: RAGState):
    q_words = {
        word.strip('.,?!').lower()
        for word in state['question'].split()
        if len(word) > 3
    }

    text = ' '.join(
        d['text'].lower()
        for d in state['documents']
    )

    matched = sum(
        word in text
        for word in q_words
    )

    score = (
        matched / len(q_words)
        if q_words
        else 0.0
    )

    return {'score': score}

## 52. Conditional route after grading

The router combines evidence quality with an application-level maximum-step policy.

In [ ]:
def route_after_grade(
    state: RAGState,
) -> Literal['generate', 'rewrite']:
    if state['score'] >= 0.25:
        return 'generate'

    if state['steps'] >= 2:
        return 'generate'

    return 'rewrite'

## 53. Rewrite node

In [ ]:
def rewrite_node(state: RAGState):
    q = state['question'].lower()

    if 'return' in q:
        improved = (
            state['question']
            + ' return window delivery deadline refund policy'
        )
    elif 'refund' in q:
        improved = (
            state['question']
            + ' refund processing payment business days'
        )
    else:
        improved = (
            state['question']
            + ' support policy eligibility requirements'
        )

    return {'query': improved}

## 54. Generation node

In [ ]:
def generate_rag_answer(state: RAGState):
    context = '\n\n'.join(
        f"[{d['source_id']}] {d['text']}"
        for d in state['documents']
    )

    prompt = f'''
Answer using only the evidence below.
If evidence is insufficient, say so.
Cite source IDs.

EVIDENCE:
{context}

QUESTION:
{state['question']}
'''

    response = local_chat.invoke(prompt)

    return {
        'answer': response.content
    }

## 55. Assemble the corrective RAG graph

In [ ]:
rag_builder = StateGraph(RAGState)

rag_builder.add_node('retrieve', retrieve_node)
rag_builder.add_node('grade', grade_node)
rag_builder.add_node('rewrite', rewrite_node)
rag_builder.add_node('generate', generate_rag_answer)

rag_builder.add_edge(START, 'retrieve')
rag_builder.add_edge('retrieve', 'grade')

rag_builder.add_conditional_edges(
    'grade',
    route_after_grade,
    {
        'generate': 'generate',
        'rewrite': 'rewrite',
    },
)

rag_builder.add_edge('rewrite', 'retrieve')
rag_builder.add_edge('generate', END)

rag_graph = rag_builder.compile(
    name='corrective_rag'
)

In [ ]:
print(rag_graph.get_graph().draw_mermaid())

## 56. Run the RAG graph

In [ ]:
rag_result = rag_graph.invoke(
    {
        'question': 'Can I return a standard item after 20 days?',
        'query': 'Can I return a standard item after 20 days?',
        'documents': [],
        'score': 0.0,
        'steps': 0,
        'answer': '',
    },
    {
        'recursion_limit': 10
    },
)

print(rag_result['answer'])
print('Steps:', rag_result['steps'])
print('Score:', round(rag_result['score'], 3))
print('Sources:', [d['source_id'] for d in rag_result['documents']])

## 57. Stream the RAG trajectory

`updates` makes the decision path visible without adding trace fields to graph state.

In [ ]:
for update in rag_graph.stream(
    {
        'question': 'What is the normal return deadline?',
        'query': 'What is the normal return deadline?',
        'documents': [],
        'score': 0.0,
        'steps': 0,
        'answer': '',
    },
    {
        'recursion_limit': 10
    },
    stream_mode='updates',
):
    print(update)

# Part XVIII — Debugging and Design Practices

## 58. Debugging order

When a graph fails, inspect it in this order:

1. **State schema** — does it contain the right raw information?
2. **Individual node** — does it produce the expected partial update?
3. **Reducer** — are repeated/parallel updates merged correctly?
4. **Router** — can every intended route actually be returned?
5. **Edges** — does the topology match the intended process?
6. **Stop condition** — can loops exit?
7. **Persistence** — are you using the intended `thread_id`?
8. **External dependencies** — model, DB, API, retriever.

Do not change the whole graph when one node contract is wrong.

## 59. Test nodes independently

Nodes are normal Python functions, so unit-test them before compiling the graph.

Example:

In [ ]:
test_state = {
    'number': 10,
    'route': '',
    'result': '',
}

print(inspect_number(test_state))
print(even_node({
    **test_state,
    'route': 'even',
}))

## 60. Keep state raw

Prefer:

```python
state['documents'] = raw_documents
state['classification'] = {'topic': 'billing', 'urgency': 'high'}
```

rather than storing a giant preformatted prompt.

Why?

- different nodes can format the data differently,
- prompts can change without state migrations,
- debugging is clearer,
- checkpoint history remains meaningful.

## 61. Separate failure types

| Failure type | Better strategy |
|---|---|
| Temporary network/API failure | `RetryPolicy` |
| Bad retrieval | corrective loop / rewrite |
| Missing user information | `interrupt()` |
| Authorization decision | deterministic rule / human review |
| Unknown programming error | let it surface for debugging |

Do not solve every failure by looping back to the LLM.

## 62. Node size

A good node usually represents one meaningful step.

Separate nodes when:

- failure/retry behavior differs,
- you need visibility between operations,
- a human may interrupt between steps,
- operations call different external systems.

Combine operations when:

- they are inseparable,
- they always fail/retry together,
- splitting them adds no useful visibility.

## 63. `Command` or edges?

Prefer normal edges/conditional edges when topology should be visible outside node code.

Prefer `Command` when a node's result naturally contains both:

- a state update, and
- a navigation decision.

Consistency matters more than choosing one style universally.

## 64. Checkpointer vs. store

These solve different persistence problems.

- **Checkpointer:** versions graph state for a thread/run.
- **Store:** persistent application memory/data that may be shared across threads.

For example:

- current conversation messages → checkpointer,
- durable user preference shared across sessions → store.

## 65. Common mistakes

### Mistake 1: Building the graph before designing state
Start from the process and shared data.

### Mistake 2: Returning the entire state from every node
Return only changed keys unless there is a specific reason not to.

### Mistake 3: Parallel nodes write the same key without a reducer
Define merge semantics explicitly.

### Mistake 4: Infinite correction loops
Use application-level maximum steps plus `recursion_limit`.

### Mistake 5: Side effects before `interrupt()`
The node re-runs from the beginning after resume.

### Mistake 6: Using one `thread_id` for unrelated users/runs
Threads are persistence boundaries.

### Mistake 7: Putting every variable into state
State should be intentional shared memory.

### Mistake 8: Using LLM routing when a normal `if` statement is sufficient
Deterministic routing is cheaper and easier to test.

# Part XIX — LangGraph API Cheat Sheet

## 66. Graph-building methods

| API | Purpose |
|---|---|
| `StateGraph(State)` | Create graph builder |
| `.add_node(name, fn)` | Register node |
| `.add_edge(a, b)` | Add unconditional edge |
| `.add_conditional_edges(source, router, map)` | Add conditional routing |
| `.set_entry_point(node)` | Entry shortcut equivalent to edge from `START` |
| `.set_finish_point(node)` | Finish shortcut equivalent to edge to `END` |
| `.set_node_defaults(...)` | Apply default retry/cache/timeout/error policies |
| `.compile(...)` | Produce executable graph |

## 67. Execution methods

| API | Purpose |
|---|---|
| `.invoke(input, config)` | Run synchronously and return final state |
| `.ainvoke(...)` | Async run |
| `.stream(...)` | Stream execution |
| `.astream(...)` | Async stream |
| `.batch(...)` | Run several inputs |
| `.get_state(config)` | Read current checkpointed state |
| `.get_state_history(config)` | Iterate state history |
| `.update_state(config, values)` | Add a manual state update |
| `.get_graph()` | Inspect compiled graph structure |

## 68. Core types

| Type | Purpose |
|---|---|
| `START` | Virtual entry node |
| `END` | Virtual finish node |
| `Command` | Update state / navigate / resume |
| `Send` | Dynamically invoke a node with custom state |
| `RetryPolicy` | Configure node retries |
| `MessagesState` | Predefined message-list state |
| `ToolNode` | Execute tool calls |
| `Runtime` | Access run context/store/stream writer |
| `StateSnapshot` | Checkpointed graph-state view |

## 69. Streaming modes to remember

| Mode | Main use |
|---|---|
| `updates` | Observe node outputs |
| `values` | Observe whole state |
| `messages` | LLM token/message stream |
| `custom` | Application progress events |
| `checkpoints` | Persistence events |
| `tasks` | Task start/finish events |
| `debug` | Maximum diagnostic detail |

## 70. Suggested learning order

1. `TypedDict` state
2. nodes returning partial updates
3. `START` / `END`
4. `add_node()`
5. `add_edge()`
6. `.compile()` and `.invoke()`
7. conditional edges
8. reducers
9. parallel branches
10. cycles + `recursion_limit`
11. `Command`
12. `Send`
13. streaming
14. checkpointers + `thread_id`
15. state inspection/update
16. `interrupt()`
17. runtime context
18. retries
19. `MessagesState`
20. `ToolNode`
21. subgraphs
22. Functional API
23. LLM/RAG workflows

## 71. Official references

- LangGraph overview: https://docs.langchain.com/oss/python/langgraph/overview
- LangGraph Graph API: https://docs.langchain.com/oss/python/langgraph/graph-api
- LangGraph reference: https://reference.langchain.com/python/langgraph
- `StateGraph`: https://reference.langchain.com/python/langgraph/graph/state/StateGraph
- `Command`: https://reference.langchain.com/python/langgraph/types/Command
- `Send`: https://reference.langchain.com/python/langgraph/types/Send
- `interrupt`: https://reference.langchain.com/python/langgraph/types/interrupt
- `ToolNode`: https://reference.langchain.com/python/langgraph.prebuilt/tool_node/ToolNode
- `MessagesState`: https://reference.langchain.com/python/langgraph/graph/message/MessagesState

Use conceptual documentation to learn architecture and the reference documentation to check exact signatures and parameters.